# Phase 3A: Feature Engineering Discovery & Exploratory Analysis

**Document Version:** 1.0.0  
**Date:** September 29, 2026  
**Status:** Feature Discovery & Design Complete (STOP Condition Enforced — Zero Model Training)  
**Dataset:** `df_train_normal` (`21c851bc-384f-5b81-8747-6dcacdceff35` / `20260225_normal`, 11,991 rows)  

> [!IMPORTANT]
> **ANTI-LEAKAGE & SCOPE CERTIFICATION:**
> 1. **Zero Model Training:** No neural networks or anomaly scoring models are trained in this phase.
> 2. **Clean Baseline Fitting Only:** All statistics, correlations, distributions, and scaling designs are derived strictly from the uncompromised normal operational baseline.
> 3. **Strict Physical Scope:** Only Photovoltaic Solar (PV) and Wind Generation channels are evaluated. Battery and Demand remain quarantined.
> 4. **Strictly Causal Transformations:** All candidate rolling statistics and temporal differences use only historical and contemporaneous timestamps ($t, t-1, \dots$). Zero lookahead leakage.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.process_data import get_default_process_extractor
from src.ml.preprocessing import SolarWindPreprocessor, FEATURE_CONFIGS

print('Imports successful! Loading normal baseline data...')

## 1. Load Clean Baseline Telemetry & Chronological Split

We extract the high-frequency causal ASOF-aligned telemetry for the clean normal baseline (`20260225_normal`) and apply the 80/20 chronological partition.

In [ ]:
extractor = get_default_process_extractor()
df_normal = extractor.extract_normal_training_set()
preprocessor = SolarWindPreprocessor(config_name='config_a', train_ratio=0.8)
train_raw, val_raw, split_info = preprocessor.chronological_split(df_normal)
train_feat, train_ts = preprocessor.extract_and_filter(train_raw)
val_feat, val_ts = preprocessor.extract_and_filter(val_raw)

print(f'Normal Baseline Samples: {len(df_normal)}')
print(f'Train Samples: {len(train_feat)}, Validation Samples: {len(val_feat)}')
print(f'Temporal gap at split boundary: {split_info["temporal_gap_at_boundary_sec"]} seconds')
print(f'Median sampling delta: {train_ts.diff().dt.total_seconds().median():.4f} seconds (~1.9 Hz)')

## 2. Statistical Inspection of the 14 Authoritative Baseline Features

We inspect the range, moments, missing values, and variance of the 14 Config A features.

In [ ]:
summary_records = []
for col in preprocessor.get_features():
    s = train_feat[col]
    summary_records.append({
        'feature': col,
        'min': s.min(),
        'p1': s.quantile(0.01),
        'median': s.median(),
        'mean': s.mean(),
        'p99': s.quantile(0.99),
        'max': s.max(),
        'std': s.std(),
        'unique': s.nunique(),
    })
pd.DataFrame(summary_records)

## 3. Category A: First-Order Temporal Differences ($\Delta x_t = x_t - x_{t-1}$)

We evaluate first differences on continuous active signals. Because sampling interval $\Delta t \approx 0.5262\text{s} \pm 0.0033\text{s}$ is uniform, $\Delta x_t = x_t - x_{t-1}$ is numerically stable and avoids timestamp jitter.

In [ ]:
diff_cols = ['pv_m_inverter_ac_power', 'wind_m_power', 'wind_m_wind_speed_a', 'pv_m_cell_temperature']
diff_df = pd.DataFrame(index=train_feat.index)
for c in diff_cols:
    diff_df[f'diff_{c}'] = train_feat[c].diff().fillna(0.0)

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for i, c in enumerate(diff_cols):
    axes[i].plot(train_ts[:400], diff_df[f'diff_{c}'][:400], color='tab:blue', lw=1.2)
    axes[i].set_ylabel(f'Δ {c.split("_")[-1]}')
    axes[i].grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (Normal Baseline)')
plt.suptitle('First-Order Temporal Differences (First 400 Samples)')
plt.tight_layout()
plt.show()

## 4. Category B & C: Causal Rolling Statistics & Local Volatility

We inspect causal rolling windows: $W=5$ (~2.6s), $W=15$ (~7.9s), $W=30$ (~15.8s), $W=60$ (~31.6s).  
**Key Finding:** Causal rolling means have $r > 0.99$ with raw signals (collinear redundancy). In contrast, rolling standard deviation (volatility) has low correlation ($r \approx 0.15 - 0.38$) and provides orthogonal information.

In [ ]:
sig = 'wind_m_power'
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(train_ts[100:500], train_feat[sig][100:500], label='Raw wind_m_power', color='black', lw=1)
for W, color in zip([5, 15, 30], ['blue', 'green', 'orange']):
    axes[0].plot(train_ts[100:500], train_feat[sig].rolling(W, min_periods=1).mean()[100:500],
                 label=f'Rolling Mean W={W}', color=color, alpha=0.7)
axes[0].set_ylabel('Power (W)')
axes[0].legend(loc='upper right')
axes[0].set_title('Rolling Means (High Collinearity with Raw Signal)')

for W, color in zip([5, 15, 30], ['blue', 'green', 'orange']):
    axes[1].plot(train_ts[100:500], train_feat[sig].rolling(W, min_periods=1).std()[100:500],
                 label=f'Rolling Std W={W}', color=color, lw=1.2)
axes[1].set_ylabel('Std (W)')
axes[1].legend(loc='upper right')
axes[1].set_title('Short-Term Volatility (Orthogonal Variance Signal)')
axes[1].set_xlabel('Time')
plt.tight_layout()
plt.show()

## 5. Category E & G: Cross-Feature Relationships & Physical Consistency

Physical invariants are the most powerful cyber-physical anomaly signals:
1. **Inverter Loss Residual ($R_{\text{inv}} = P_{dc} - P_{ac}$):** Preserves electrical energy conservation.
2. **Dual Anemometer Delta ($\Delta v = v_b - v_a$):** Detects single-anemometer register spoofing.
3. **Dual Nacelle Temp Delta ($\Delta T = T_b - T_a$):** Enforces thermal symmetry.
4. **PV Cell Thermal Gradient ($\Delta T = T_{\text{cell}} - T_{\text{air}}$):** Validates photoelectric solar heating.

In [ ]:
res_inv_loss = train_feat['pv_m_inverter_dc_power'] - train_feat['pv_m_inverter_ac_power']
diff_anemometer = train_feat['wind_m_wind_speed_b'] - train_feat['wind_m_wind_speed_a']
diff_nac_temp = train_feat['wind_m_temperature_b'] - train_feat['wind_m_temperature_a']
diff_pv_thermal = train_feat['pv_m_cell_temperature'] - train_feat['pv_m_temp_air']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].hist(res_inv_loss, bins=35, color='teal', edgecolor='black', alpha=0.7)
axes[0, 0].set_title(f'Inverter Loss P_dc - P_ac (Mean={res_inv_loss.mean():.1f}W, Std={res_inv_loss.std():.1f}W)')

axes[0, 1].hist(diff_anemometer, bins=35, color='green', edgecolor='black', alpha=0.7)
axes[0, 1].set_title(f'Anemometer Delta v_b - v_a (Mean={diff_anemometer.mean():.2f}m/s, Std={diff_anemometer.std():.2f}m/s)')

axes[1, 0].hist(diff_nac_temp, bins=35, color='darkorange', edgecolor='black', alpha=0.7)
axes[1, 0].set_title(f'Nacelle Temp Delta T_b - T_a (Mean={diff_nac_temp.mean():.3f}°C, Std={diff_nac_temp.std():.3f}°C)')

axes[1, 1].scatter(train_feat['pv_m_poa_direct'], diff_pv_thermal, alpha=0.2, s=8, color='purple')
axes[1, 1].set_xlabel('Direct Solar Irradiance (W/m²)')
axes[1, 1].set_ylabel('Cell Thermal Rise (°C)')
axes[1, 1].set_title('Cell Thermal Rise vs Irradiance')
plt.tight_layout()
plt.show()

## 6. Redundancy Analysis & Candidate Correlations

We inspect pairwise correlations across candidate features to reject redundant near-duplicates and retain high-signal, physically grounded representations.

In [ ]:
pool = train_feat.copy()
pool['res_inv_loss'] = res_inv_loss
pool['diff_anemometer_ab'] = diff_anemometer
pool['diff_nac_temp_ab'] = diff_nac_temp
pool['diff_pv_cell_thermal'] = diff_pv_thermal
pool['diff_pv_ac_power'] = train_feat['pv_m_inverter_ac_power'].diff().fillna(0.0)
pool['diff_wind_power'] = train_feat['wind_m_power'].diff().fillna(0.0)
pool['roll_std15_wind_power'] = train_feat['wind_m_power'].rolling(15, min_periods=1).std().fillna(0.0)

corr_matrix = pool.corr()
plt.figure(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Candidate Feature Redundancy Heatmap (Normal Baseline)')
plt.tight_layout()
plt.show()

## 7. Proposed Feature Engineering Configurations

We define four controlled configurations for future comparative experimentation:
- **CONFIG A (Baseline):** 14 raw features.
- **CONFIG B (Temporal Dynamics):** 19 features (14 raw + 3 differences + 2 rolling volatilities).
- **CONFIG C (Physical Invariants):** 18 features (14 raw + 4 physical residuals).
- **CONFIG D (Recommended Parsimonious Hybrid):** 22 features (14 raw + 4 physical residuals + 3 differences + 1 persistence feature).

> [!NOTE]
> **STOP Condition:** In strict adherence to project requirements, no neural network is trained in this phase. The pipeline design is saved for operator and stakeholder review.